# k-Nearest Neighbours, Naive Bayes and Support Vector Machines – Concepts

This notebook summarises the concepts and adds explanations and small examples. All examples use small hand-made numbers or simulated data – no data set is loaded.

**Contents**
1. k-nearest neighbours (k-NN)
2. Scaling of features
3. Naive Bayes (worked example: play golf?)
4. Naive Bayes for text
5. Data leakage
6. Support vector machines (SVM)

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_blobs, make_circles
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer

# Ignore warnings
warnings.filterwarnings('ignore')

## 1. k-nearest neighbours (k-NN)

1. Choose $k$.
2. Compute the distances of the new point $x$ to all training points.
3. Take the $k$ nearest: $\mathcal N_k(x)$.
4. Predict the majority class (classification) or the mean of $y$ (regression).

**Distance** (Euclidean – straight-line distance, as with a ruler):

$$d(x, x') = \sqrt{\sum_{j=1}^{p}(x_j - x'_j)^2}$$

**Class probability** = share of the $k$ nearest neighbours that belong to class $c$:
$\hat P(y = c \mid x) = \frac{1}{k}\sum_{i \in \mathcal N_k(x)} 1(y_i = c)$. Example: $k = 5$, three neighbours blue →
P(blue) = 3/5 = 0.6.

**Choice of $k$:** small $k$ → flexible, wiggly boundary (high variance); large $k$ → smooth boundary (high bias).
Choose $k$ by cross-validation. k-NN has no real training phase ("lazy learner") – all the work happens at prediction
time.

In [ ]:
# k-NN decision boundaries for k = 1 and k = 15
X, y = make_moons(n_samples=250, noise=0.35, random_state=0)
xx, yy = np.meshgrid(np.linspace(-2, 3, 300), np.linspace(-1.5, 2, 300))

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, k, title in [
    (axes[0], 1, 'k = 1 (high variance)'),
    (axes[1], 15, 'k = 15 (smoother)'),
]:
    knn = KNeighborsClassifier(n_neighbors=k).fit(X, y)
    zz = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(
        xx,
        yy,
        zz,
        levels=[-0.5, 0.5, 1.5],
        colors=['tab:orange', 'tab:blue'],
        alpha=0.2,
    )
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=12, color='tab:orange')
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=12, color='tab:blue')
    ax.set_title(title)
plt.show()

## 2. Scaling of features

**Caution:** scale the features when their units are not comparable – otherwise the variable with the largest scale
dominates the distance. Standardisation: $z_j = (x_j - \bar x_j)/s_j$.

Example: two persons differ by 30 years in age and by 300 CHF in income. Without scaling, the income difference
dominates completely; after standardising (e.g. $s_{\text{age}} = 15$ years, $s_{\text{income}} = 2'000$ CHF), the age
difference (2 standard deviations) matters much more than the income difference (0.15 standard deviations).

In scikit-learn the scaler belongs into a **pipeline** (`make_pipeline(StandardScaler(), KNeighborsClassifier())`), so
that it is fitted on the training data only. The same applies to SVMs.

In [ ]:
# Distance with and without standardisation
diff = np.array([30, 300])  # difference in age [years] and income [CHF]
sd = np.array([15, 2000])
print(
    'distance unscaled   :',
    round(np.sqrt(np.sum(diff**2)), 1),
    ' (dominated by income)',
)
print(
    'distance standardised:',
    round(np.sqrt(np.sum((diff / sd) ** 2)), 2),
    ' (dominated by age)',
)

## 3. Naive Bayes

**Bayes' theorem** (Thomas Bayes, 1701–1761):

$$P(y \mid x) = \frac{P(x \mid y)\,P(y)}{P(x)}$$

In words: how likely is class $y$ after seeing $x$? Start from the **base rate** $P(y)$ and update it by how typical $x$
is for $y$.

**"Naive": the features are assumed to be independent given the class:**

$$P(y \mid x_1, \dots, x_p) \;\propto\; P(y)\prod_{j=1}^{p} P(x_j \mid y)$$

In words: multiply the base rate with one probability per feature; predict the class with the highest score.

**Worked example: play golf?** New day: outlook = sunny, temperature = cool, humidity = high, windy = true.

In [ ]:
# Naive Bayes by hand: play golf?
golf = pd.DataFrame(
    {
        'outlook': [
            'sunny',
            'sunny',
            'overcast',
            'rainy',
            'rainy',
            'rainy',
            'overcast',
            'sunny',
            'sunny',
            'rainy',
            'sunny',
            'overcast',
            'overcast',
            'rainy',
        ],
        'temp': [
            'hot',
            'hot',
            'hot',
            'mild',
            'cool',
            'cool',
            'cool',
            'mild',
            'cool',
            'mild',
            'mild',
            'mild',
            'hot',
            'mild',
        ],
        'humidity': [
            'high',
            'high',
            'high',
            'high',
            'normal',
            'normal',
            'normal',
            'high',
            'normal',
            'normal',
            'normal',
            'high',
            'normal',
            'high',
        ],
        'windy': [
            False,
            True,
            False,
            False,
            False,
            True,
            True,
            False,
            False,
            False,
            True,
            True,
            False,
            True,
        ],
        'play': [
            'no',
            'no',
            'yes',
            'yes',
            'yes',
            'no',
            'yes',
            'no',
            'yes',
            'yes',
            'yes',
            'yes',
            'yes',
            'no',
        ],
    }
)
print(golf.to_string())

new_day = {
    'outlook': 'sunny',
    'temp': 'cool',
    'humidity': 'high',
    'windy': True,
}
scores = {}
for c in ['yes', 'no']:
    sub = golf[golf['play'] == c]
    score = len(sub) / len(golf)  # base rate P(y)
    factors = [f'{len(sub)}/{len(golf)}']
    for feature, value in new_day.items():
        k = (sub[feature] == value).sum()
        score *= k / len(sub)  # P(x_j | y)
        factors.append(f'{k}/{len(sub)}')
    scores[c] = score
    print(f'score({c}) = ' + ' * '.join(factors) + f' = {score:.4f}')
print(f'P(no | x) = {scores["no"] / (scores["no"] + scores["yes"]):.3f}')

**Result:** score(no) = 0.0206 > score(yes) = 0.0053 → predict "no". Normalised: P(no | x) = 0.0206 / (0.0206 + 0.0053)
≈ 0.795.

**Additional explanation.** If a value never occurs in a class (e.g. *overcast* never occurs on "no" days), the product
becomes 0. **Laplace smoothing** adds a small count $\alpha$ (usually 1) to every count:
$P(x_j = v \mid y) = \frac{n_{v,y} + \alpha}{n_y + \alpha\,V_j}$ ($V_j$ = number of possible values). Despite the
unrealistic independence assumption, Naive Bayes often classifies well – but its probabilities are often too extreme.

## 4. Naive Bayes for text

Texts are turned into word counts (**bag of words**): each column is a word of the vocabulary, each row a document; word
order is ignored. `MultinomialNB` estimates $P(\text{word} \mid y)$ from the word counts per class (with Laplace
smoothing) and multiplies them for all words of a document.

The ratio $\log P(\text{word} \mid \text{class } 1) - \log P(\text{word} \mid \text{class } 0)$ shows which words are
typical for each class.

In [ ]:
# Naive Bayes for text: spam vs. normal messages
docs = [
    'free prize call now',
    'win a free prize',
    'call me later',
    'lunch tomorrow',
    'are you free for lunch',
    'claim your free cash prize',
]
labels = [1, 1, 0, 0, 0, 1]  # 1 = spam, 0 = normal message

vec = CountVectorizer()
W = vec.fit_transform(docs)
print(
    pd.DataFrame(
        W.toarray(), columns=vec.get_feature_names_out(), index=docs
    ).to_string()
)

nb = MultinomialNB().fit(W, labels)
log_ratio = pd.Series(
    nb.feature_log_prob_[1] - nb.feature_log_prob_[0],
    index=vec.get_feature_names_out(),
)
print('\nlog ratio (spam vs. normal):')
print(log_ratio.sort_values(ascending=False).round(2).to_string())
print(
    '\nP(spam | "free lunch") =',
    round(nb.predict_proba(vec.transform(['free lunch']))[0, 1], 3),
)

## 5. Data leakage

**Data leakage** means that a model uses information that would **not be available at the time of the prediction** –
often information that is created *after* the outcome or that contains the target itself (e.g. a text field describing
the outcome, a variable measured after the event, or preprocessing fitted on the test data). The model then looks
excellent in the evaluation but fails in practice.

Checklist: for every feature ask *"Do I know this value at the moment I want to make the prediction?"* – and fit all
preprocessing steps (scaling, vocabulary of a text vectorizer, imputation) on the training data only.

## 6. Support vector machines (SVM)

- **Widest gap:** find the line (hyperplane $w^\top x + b = 0$) that separates the classes with the **largest margin**.
- **Support vectors:** the points closest to the line determine it – all others do not matter.
- **$C$ (soft margin):** with a soft margin, support vectors may lie inside the gap or even be misclassified. Small $C$:
  wide gap, some errors allowed; large $C$: few errors, narrow gap (risk of overfitting).
- **Kernel:** measures similarity in a transformed space → curved boundaries (kernel trick), e.g. the RBF kernel
  $K(x, x') = \exp(-\gamma\lVert x - x'\rVert^2)$.

Formally, the soft-margin SVM minimises hinge loss + penalty:
$\frac{1}{2}\lVert w\rVert^2 + C\sum_i \max\big(0, 1 - y_i(w^\top x_i + b)\big)$.

Interactive demo: jgreitemann.github.io/svm-demo

In [ ]:
# SVM with a linear and an RBF kernel on simulated data
X_lin, y_lin = make_blobs(
    n_samples=60, centers=[[-2, -1], [2, 1.5]], cluster_std=0.9, random_state=3
)
X_circ, y_circ = make_circles(
    n_samples=200, noise=0.08, factor=0.4, random_state=0
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (Xs, ys, kernel, title) in zip(
    axes,
    [
        (X_lin, y_lin, 'linear', 'Linear SVM: maximal margin'),
        (X_circ, y_circ, 'rbf', 'RBF kernel: non-linear boundary'),
    ],
):
    svm = SVC(kernel=kernel, C=1).fit(Xs, ys)
    gx, gy = np.meshgrid(
        np.linspace(Xs[:, 0].min() - 0.5, Xs[:, 0].max() + 0.5, 300),
        np.linspace(Xs[:, 1].min() - 0.5, Xs[:, 1].max() + 0.5, 300),
    )
    zz = svm.decision_function(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(
        gx,
        gy,
        zz > 0,
        levels=[-0.5, 0.5, 1.5],
        colors=['tab:orange', 'tab:blue'],
        alpha=0.15,
    )
    ax.contour(
        gx,
        gy,
        zz,
        levels=[-1, 0, 1],
        colors='black',
        linestyles=['--', '-', '--'],
        linewidths=[1, 2, 1],
    )
    ax.scatter(Xs[ys == 0, 0], Xs[ys == 0, 1], s=15, color='tab:orange')
    ax.scatter(Xs[ys == 1, 0], Xs[ys == 1, 1], s=15, color='tab:blue')
    sv = svm.support_vectors_
    ax.scatter(
        sv[:, 0],
        sv[:, 1],
        s=90,
        facecolors='none',
        edgecolors='darkred',
        linewidths=1.2,
    )
    ax.set_title(f'{title} ({len(sv)} support vectors)')
plt.show()

In [ ]:
# Effect of C on the margin and the number of support vectors
# (linear kernel, overlapping classes)
X_ov, y_ov = make_blobs(
    n_samples=100,
    centers=[[-1, -0.5], [1, 0.5]],
    cluster_std=1.0,
    random_state=1,
)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, C in zip(axes, [0.01, 1, 100]):
    svm = SVC(kernel='linear', C=C).fit(X_ov, y_ov)
    gx, gy = np.meshgrid(np.linspace(-4, 4, 200), np.linspace(-3.5, 3.5, 200))
    zz = svm.decision_function(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contour(
        gx,
        gy,
        zz,
        levels=[-1, 0, 1],
        colors='black',
        linestyles=['--', '-', '--'],
    )
    ax.scatter(X_ov[y_ov == 0, 0], X_ov[y_ov == 0, 1], s=12, color='tab:orange')
    ax.scatter(X_ov[y_ov == 1, 0], X_ov[y_ov == 1, 1], s=12, color='tab:blue')
    ax.scatter(
        *svm.support_vectors_.T, s=70, facecolors='none', edgecolors='darkred'
    )
    ax.set_title(f'C = {C}: {len(svm.support_)} support vectors')
plt.show()

**Reading the plots:** the solid line is the decision boundary, the dashed lines the margin. Only the circled support
vectors determine the boundary. A small $C$ gives a wide margin with many support vectors (more bias), a large $C$ a
narrow margin with few support vectors (more variance).

**Key messages**
- k-NN and SVM are based on distances – **scale the features**.
- Naive Bayes multiplies the base rate with one probability per feature; it is simple, fast and works well for text.
- An SVM is determined by its support vectors; $C$ and the kernel control the flexibility.
- Always check whether a feature would be available at prediction time (data leakage).

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')